## A.4. Noise sensitivity

Assess dynamical and observational noise separately with paired data and reservoir seeds. Each condition uses 20 ESNs, 30 histories per model and five evolution steps.

Existing datasets, trained models and DDI checkpoints are reused. Missing data and model results are computed.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
import gc
import time
import tempfile
from matplotlib.ticker import MaxNLocator
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
%run ./time_Lorenz_data_6time.ipynb
%run ./ESN.ipynb
%run ./Reconstruct.ipynb
%run ./Plot_DDI.ipynb
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['font.size'] = 16
plt.rcParams['axes.titlesize'] = 17
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['xtick.labelsize'] = 13
plt.rcParams['ytick.labelsize'] = 13
plt.rcParams['legend.fontsize'] = 12
N_x = 4
N_washout = 100
N_train = 10000
N_tstart = 10000
N_test = 2000
window = 6
window_length = 100
test_window = 6
lag_list = np.arange(test_window) * window_length
N_net = 20
N_rep = 30
N_evo = 5
c = 0.3
dt = 0.01
sigma_dyn_values = np.array([0.0, 1e-10, 1e-07, 0.0001, 0.2, 0.4, 0.7, 1.0, 1.4])
sigma_obs_values = np.array([0.0, 0.0001, 0.001, 0.005, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06])
FIXED_SIGMA_DYN = 0.0001
FIXED_SIGMA_OBS = 0.001
TRUE_LAG_INDEX = 3
TRUE_EDGE = (0, 1)
FALSE_EDGE = (1, 0)
DATA_SEED = 202603
OFFSET_SEED = 202604
RESERVOIR_SEED = 202605
DATA_PATH = os.path.join('result', 'A.4', 'A.4_data.pkl')
RESULT_PATH = os.path.join('result', 'A.4', 'A.4.pkl')
FIGURE_OBS_PATH = os.path.join('result', 'figure', 'A.4_sigma_obs.png')
FIGURE_DYN_PATH = os.path.join('result', 'figure', 'A.4_sigma_dyn.png')
ESN_DIR = os.path.join('result', 'A.4', 'ESN')
os.makedirs(ESN_DIR, exist_ok=True)
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
os.makedirs(os.path.dirname(FIGURE_OBS_PATH), exist_ok=True)


### Part 1. Load the noise conditions

Reuse the saved datasets. Missing conditions use the same network and paired random seeds, with only the selected noise level varied.


In [ ]:
A_list = [np.zeros((N_x, N_x), dtype=float) for _ in range(test_window)]
A_list[TRUE_LAG_INDEX][TRUE_EDGE] = 1.0
A_tensor = np.stack(A_list, axis=0)

def condition_key(sigma_dyn, sigma_obs):
    return f'dyn_{sigma_dyn:.12e}_obs_{sigma_obs:.12e}'
obs_conditions = [(FIXED_SIGMA_DYN, float(sigma_obs)) for sigma_obs in sigma_obs_values]
dyn_conditions = [(float(sigma_dyn), FIXED_SIGMA_OBS) for sigma_dyn in sigma_dyn_values]
sweep_keys = {'sigma_obs': [condition_key(*item) for item in obs_conditions], 'sigma_dyn': [condition_key(*item) for item in dyn_conditions]}
experiment_signature = {'N_x': N_x, 'N_tstart': N_tstart, 'N_test': N_test, 'window': window, 'window_length': window_length, 'c': c, 'dt': dt, 'sigma_dyn_values': tuple(sigma_dyn_values.tolist()), 'sigma_obs_values': tuple(sigma_obs_values.tolist()), 'FIXED_SIGMA_DYN': FIXED_SIGMA_DYN, 'FIXED_SIGMA_OBS': FIXED_SIGMA_OBS, 'DATA_SEED': DATA_SEED}
datasets = {}
if os.path.exists(DATA_PATH):
    with open(DATA_PATH, 'rb') as f:
        previous_data = pickle.load(f)
    if previous_data.get('experiment_signature') == experiment_signature:
        datasets.update(previous_data.get('datasets', {}))
unique_conditions = sorted(set(obs_conditions + dyn_conditions))
n_data = N_tstart + N_test
for (sigma_dyn, sigma_obs) in unique_conditions:
    key = condition_key(sigma_dyn, sigma_obs)
    if key in datasets:
        continue
    np.random.seed(DATA_SEED)
    X = gen_data_norm_6time(N_x, *A_list, n_data, window, window_length, c=c, sigma_dyn=sigma_dyn, sigma_obs=sigma_obs, dt=dt)
    datasets[key] = X
    data_bundle = {'A_tensor': A_tensor, 'datasets': datasets, 'sweep_keys': sweep_keys, 'obs_conditions': obs_conditions, 'dyn_conditions': dyn_conditions, 'experiment_signature': experiment_signature}
    with open(DATA_PATH, 'wb') as f:
        pickle.dump(data_bundle, f)


### Part 2. Compute DDI and plot

Average the full DDI matrices across 20 ESNs, then summarize true and absent edges. Errors propagate directly. Completed model and DDI files remain available for resuming and redrawing the PNG figures.


In [ ]:
with open(DATA_PATH, 'rb') as f:
    data_bundle = pickle.load(f)
if data_bundle.get('experiment_signature') != experiment_signature:
    raise RuntimeError('The saved datasets do not match the current configuration.')
datasets = data_bundle['datasets']
sweep_keys = data_bundle['sweep_keys']
missing_keys = [key for key in sweep_keys['sigma_obs'] + sweep_keys['sigma_dyn'] if key not in datasets]
if missing_keys:
    raise RuntimeError('Part 1 is incomplete. Missing: ' + ', '.join(missing_keys))
N_dim = 3 * N_x
N_dim_segments = test_window * N_dim
N_units_segments = 100 * N_dim_segments
max_lag = int(lag_list.max())

def build_delayed_dataset(X):
    (T, D) = X.shape
    X_segments = np.zeros((T - max_lag, test_window * D))
    for (row, absolute_index) in enumerate(range(max_lag, T)):
        X_segments[row] = np.concatenate([X[absolute_index - lag] for lag in lag_list])
    X_washout_segments = X_segments[:N_washout]
    X_t_segments = X_segments[N_washout:N_washout + N_train - 1]
    Y_t_segments = X_segments[N_washout + 1:N_washout + N_train]
    X_test_segments = X_segments[N_tstart:]
    return (X_washout_segments, X_t_segments, Y_t_segments, X_test_segments)
reference_X = datasets[sweep_keys['sigma_obs'][0]]
reference_test_length = build_delayed_dataset(reference_X)[-1].shape[0]
max_offset = reference_test_length - N_evo - 1
if max_offset <= N_washout:
    raise ValueError('The test sequence is too short for the requested offsets.')
offset_rng = np.random.RandomState(OFFSET_SEED)
offset = offset_rng.randint(N_washout, max_offset, size=(N_net, N_rep))
DDI_matrices = {'sigma_obs': np.full((len(sigma_obs_values), N_net, N_x, N_x), np.nan), 'sigma_dyn': np.full((len(sigma_dyn_values), N_net, N_x, N_x), np.nan)}
result_signature = {**experiment_signature, 'N_washout': N_washout, 'N_train': N_train, 'N_net': N_net, 'N_rep': N_rep, 'N_evo': N_evo, 'OFFSET_SEED': OFFSET_SEED, 'RESERVOIR_SEED': RESERVOIR_SEED, 'TRUE_LAG_INDEX': TRUE_LAG_INDEX, 'N_units_segments': N_units_segments}

def save_pickle_atomic(value, path):
    path = Path(path)
    (descriptor, temporary_path) = tempfile.mkstemp(dir=path.parent, suffix='.tmp')
    with os.fdopen(descriptor, 'wb') as stream:
        pickle.dump(value, stream, protocol=pickle.HIGHEST_PROTOCOL)
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temporary_path, path)
model_paths = {sweep_name: [[os.path.join(ESN_DIR, f'{key}_net_{net:02d}.pkl') for net in range(N_net)] for key in sweep_keys[sweep_name]] for sweep_name in DDI_matrices}
if os.path.exists(RESULT_PATH):
    with open(RESULT_PATH, 'rb') as f:
        previous_result = pickle.load(f)
    if previous_result.get('result_signature') == result_signature:
        for sweep_name in DDI_matrices:
            previous_matrices = np.asarray(previous_result['DDI_matrices'][sweep_name])
            if previous_matrices.shape == DDI_matrices[sweep_name].shape:
                DDI_matrices[sweep_name][:] = previous_matrices
for (sweep_name, noise_values) in (('sigma_obs', sigma_obs_values), ('sigma_dyn', sigma_dyn_values)):
    for (condition_idx, key) in enumerate(sweep_keys[sweep_name]):
        X = datasets[key]
        (X_washout_segments, X_t_segments, Y_t_segments, X_test_segments) = build_delayed_dataset(X)
        for net in range(N_net):
            model_path = model_paths[sweep_name][condition_idx][net]
            matrix_complete = np.all(np.isfinite(DDI_matrices[sweep_name][condition_idx, net]))
            if matrix_complete:
                continue
            if 'esn' in globals():
                del esn
            gc.collect()
            model_loaded = False
            if os.path.exists(model_path):
                with open(model_path, 'rb') as f:
                    model_bundle = pickle.load(f)
                if model_bundle.get('result_signature') == result_signature:
                    esn = model_bundle['esn']
                    model_loaded = True
                    del model_bundle
            if not model_loaded:
                np.random.seed(RESERVOIR_SEED + net)
                esn = ESN(N_units_segments, N_dim_segments, N_dim_segments)
                esn.train(X_washout_segments, X_t_segments, Y_t_segments)
                save_pickle_atomic({'esn': esn, 'result_signature': result_signature, 'condition_key': key, 'net': net, 'offset': offset[net]}, model_path)
            ddi_path = model_path.replace('.pkl', '_DDI.pkl')
            DDI = None
            if os.path.exists(ddi_path):
                with open(ddi_path, 'rb') as f:
                    ddi_bundle = pickle.load(f)
                if ddi_bundle.get('result_signature') == result_signature:
                    DDI = np.asarray(ddi_bundle['DDI'])
                del ddi_bundle
            if DDI is None:
                DDI = cal_time_TDI_copy_detect(esn, X_test_segments, N_x, 3, N_dim, N_rep, test_window, 0, 1, N_washout, N_evo, offset[net], TRUE_LAG_INDEX)
                save_pickle_atomic({'DDI': DDI, 'result_signature': result_signature}, ddi_path)
            DDI_matrices[sweep_name][condition_idx, net] = DDI
            save_pickle_atomic({'DDI_matrices': DDI_matrices, 'model_paths': model_paths, 'offset': offset, 'A_tensor': data_bundle['A_tensor'], 'sigma_dyn_values': sigma_dyn_values, 'sigma_obs_values': sigma_obs_values, 'result_signature': result_signature}, RESULT_PATH)
            del esn
            gc.collect()


In [ ]:
with open(RESULT_PATH, 'rb') as f:
    saved_results = pickle.load(f)
DDI_matrices = saved_results['DDI_matrices']
sigma_obs_values = np.asarray(saved_results['sigma_obs_values'])
sigma_dyn_values = np.asarray(saved_results['sigma_dyn_values'])
FALSE_EDGE = (1, 0)
scores = {}
for (sweep_name, matrices) in DDI_matrices.items():
    matrices = np.asarray(matrices)
    if not np.all(np.isfinite(matrices)):
        raise RuntimeError('Finish all DDI computations before plotting.')
    scores[sweep_name] = np.stack([matrices[:, :, TRUE_EDGE[0], TRUE_EDGE[1]], matrices[:, :, FALSE_EDGE[0], FALSE_EDGE[1]]], axis=-1)

def format_noise_title(symbol, value):
    (coefficient, exponent) = f'{value:.12e}'.split('e')
    coefficient = float(coefficient)
    power = f'10^{{{int(exponent)}}}'
    value_label = power if coefficient == 1.0 else f'{coefficient:g}\\times {power}'
    return f'$\\sigma_{{\\mathrm{{{symbol}}}}}={value_label}$'

def plot_noise_robustness(noise_values, edge_scores, x_label, fixed_title, output_path, x_exponent):
    causal_color = '#1F77B4'
    noncausal_color = '#D55E00'
    colors = (causal_color, noncausal_color)
    labels = (f'Causal edge: {TRUE_EDGE[1] + 1} → {TRUE_EDGE[0] + 1}', f'Non-causal edge: {FALSE_EDGE[1] + 1} → {FALSE_EDGE[0] + 1}')
    mean_scores = np.nanmean(edge_scores, axis=1)
    std_scores = np.nanstd(edge_scores, axis=1, ddof=1)
    (fig, ax) = plt.subplots(figsize=(8.0, 5.6), facecolor='white')
    fig.subplots_adjust(left=0.13, right=0.97, bottom=0.16, top=0.88)
    positive_values = noise_values[noise_values > 0]
    if positive_values.size > 1:
        minimum_spacing = np.min(np.diff(np.unique(positive_values)))
    else:
        minimum_spacing = 1.0
    jitter_span = 0.12 * minimum_spacing
    jitter = np.linspace(-jitter_span, jitter_span, N_net)
    for (edge_idx, (color, label)) in enumerate(zip(colors, labels)):
        ax.plot(noise_values, mean_scores[:, edge_idx], color=color, linewidth=2.2, marker=('o', 's')[edge_idx], markersize=5.5, markeredgecolor='white', markeredgewidth=0.6, linestyle=('-', '--')[edge_idx], label=label, zorder=3)
        ax.fill_between(noise_values, np.maximum(mean_scores[:, edge_idx] - std_scores[:, edge_idx], 0.0), mean_scores[:, edge_idx] + std_scores[:, edge_idx], color=color, alpha=0.14, linewidth=0, zorder=1)
        for (condition_idx, noise_value) in enumerate(noise_values):
            ax.scatter(noise_value + jitter, edge_scores[condition_idx, :, edge_idx], s=18, color=color, alpha=0.55, edgecolors='none', zorder=2)
    ax.set_xlabel(x_label)
    ax.set_ylabel('DDI')
    ax.set_title(fixed_title, fontsize=15, pad=12)
    data_top = max(np.max(edge_scores), np.max(mean_scores + std_scores))
    ax.set_ylim(0, 1.26 * data_top)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.tick_params(axis='both', labelsize=12, width=1.0, direction='out')
    ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0), useMathText=True)
    ax.ticklabel_format(axis='x', style='sci', scilimits=(x_exponent, x_exponent), useMathText=True)
    ax.xaxis.get_offset_text().set_fontsize(11)
    ax.yaxis.get_offset_text().set_fontsize(11)
    ax.set_axisbelow(True)
    ax.grid(False)
    ax.legend(loc='upper right', bbox_to_anchor=(0.98, 0.98), frameon=False, fontsize=12, handlelength=2.6, labelspacing=0.7, borderaxespad=0.5)
    for spine in ax.spines.values():
        spine.set_linewidth(1.0)
    fig.savefig(output_path, dpi=600, facecolor='white')
    plt.show()
plot_noise_robustness(sigma_obs_values, scores['sigma_obs'], '$\\sigma_{\\mathrm{obs}}$', format_noise_title('dyn', FIXED_SIGMA_DYN), FIGURE_OBS_PATH, x_exponent=-2)
plot_noise_robustness(sigma_dyn_values, scores['sigma_dyn'], '$\\sigma_{\\mathrm{dyn}}$', format_noise_title('obs', FIXED_SIGMA_OBS), FIGURE_DYN_PATH, x_exponent=-1)
print(FIGURE_OBS_PATH)
print(FIGURE_DYN_PATH)
